### Importar modulos necessários para acessar os codigos

In [ ]:
from Treinos.Treinamento_Unet import treinar_modelo
from Testes.Testes import avaliar_e_visualizar
from Utils import DataLoader, Quantizacao
from Modelos.UNet_baseline import UNetBaseline
from Modelos.UNet_MobileNet_v2 import UNetMobileNetV2
from Modelos.UNet_MobileNet_v3 import UNetMobileNetV3
from Modelos.UNet_SegFormer import SegFormerB0
from Modelos.UNet_MobileNetV3_AttentionGates import UNetMobileNetV3AttentionGates
from Modelos.UNet_SegFormer_AttentionGates import SegFormerB0_Attention

In [ ]:
MODELS = [#(UNetBaseline, "UNet_baseline", 0),
          (UNetMobileNetV2, "Mobile_Net_v2", 0),
          (UNetMobileNetV3, "MobileNet_v3_FD", 0),
          (UNetMobileNetV3AttentionGates, "MobileNetV3_AttentionGates_FD", 0),
          (UNetMobileNetV3, "MobileNet_v3_BCE", 0),
          (UNetMobileNetV3AttentionGates, "MobileNetV3_AttentionGates_BCE", 0), 
          (SegFormerB0, "UNet_SegFormer", 0),
          (SegFormerB0_Attention, "UNet_SegFormer_AttGate_BCE", 0),
          (SegFormerB0, "UNet_SegFormer_FD", 0),
          (SegFormerB0_Attention, "UNet_SegFormer_AttGate_FD", 0)
          ]

Treinar  Mag1c + RGB

In [ ]:
for MODELO, NOME_MODELO, starting_point in MODELS:
    nome_modelo = NOME_MODELO + "_mag1c_rgb"
    treinar_modelo(MODELO, nome_modelo, starting_point,selected_loss="FD",produtos_entrada=["mag1c", "TOA_AVIRIS_460nm", "TOA_AVIRIS_550nm", "TOA_AVIRIS_640nm"])

Testar mag1c + RGB

In [ ]:
DEVICES = ["cuda","cpu"]
for device in DEVICES:
    for MODELO, NOME_MODELO, _ in MODELS:
        nome_modelo = NOME_MODELO + "_mag1c_rgb"
        avaliar_e_visualizar(MODELO, nome_modelo, ["mag1c", "TOA_AVIRIS_460nm", "TOA_AVIRIS_550nm", "TOA_AVIRIS_640nm"], device)

Calculo dos GMACs

In [ ]:
import torch

from thop import profile

def calcular_gmacs(modelo, device, canais_entrada=4, altura=512, largura=512):
    modelo.eval()
    
    # Cria um tensor vazio (dummy) com a resolução que você quer testar
    # Usamos Batch Size = 1 para padronizar o custo por imagem
    tensor_teste = torch.randn(1, canais_entrada, altura, largura).to(device)
    
    # O thop rastreia o forward pass e conta as operações
    macs, params = profile(modelo, inputs=(tensor_teste, ), verbose=False)
    
    # 1 GMAC = 1 bilhão de MACs
    gmacs = macs / 1e9
    
    print(f"\n--- Análise de Custo Computacional ---")
    print(f"Resolução de Entrada : {altura}x{largura} ({canais_entrada} canais)")
    print(f"Parâmetros Totais    : {params / 1e6:.2f} M")
    print(f"Custo Computacional  : {gmacs:.3f} GMACs")
    
    return gmacs

# Defina o dispositivo e a quantidade de canais do seu dataset atual (mag1c + 3 RGB = 4)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
canais_entrada = 4

print("=== Relatório de Custos Computacionais (Inferência) ===")

for ModeloClass, nome_modelo, _ in MODELS:
    print(f"\nAnalisando arquitetura: {nome_modelo}")
    
    # 1. Instancia o modelo do zero na GPU
    modelo_instanciado = ModeloClass(in_channels=canais_entrada, out_channels=1).to(device)
    
    # 2. Calcula os GMACs simulando a resolução final de teste (512x512)
    gmacs_512 = calcular_gmacs(
        modelo=modelo_instanciado, 
        device=device, 
        canais_entrada=canais_entrada, 
        altura=512, 
        largura=512
    )

Quantizacao

In [ ]:
import os
import torch

PRODUTOS_ENTRADA = ["mag1c", "TOA_AVIRIS_460nm", "TOA_AVIRIS_550nm", "TOA_AVIRIS_640nm"]
CAMINHO_CSV_TREINO = "Datasets/STARCOP_train/train.csv"
DIRETORIO_DADOS_TREINO = "Datasets/STARCOP_train"
DEVICE_QUANT = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Carregamento do conjunto de calibração (150 amostras do STARCOP sem rótulos)
df_train_calib = DataLoader.carregar_dataframe_starcop(CAMINHO_CSV_TREINO, DIRETORIO_DADOS_TREINO)
dataset_calib = DataLoader.STARCOPDataset(df_train_calib.iloc[:150], PRODUTOS_ENTRADA, ["labelbinary"], modo_treino=False)
dataloader_calib = DataLoader(dataset_calib, batch_size=1, shuffle=False)

normalizador_calib = DataLoader.DataNormalizer(PRODUTOS_ENTRADA).to(DEVICE_QUANT)
dummy_input = torch.randn(1, len(PRODUTOS_ENTRADA), 512, 512).to(DEVICE_QUANT)

print(" INICIANDO PIPELINE DE QUANTIZAÇÃO INT8 E TESTES DOS MODELOS")

for MODELO_CLASSE, NOME_MODELO, _ in MODELS:
    nome_modelo_fp32 = NOME_MODELO + "_mag1c_rgb"
    caminho_pesos = f"Modelos_treinados/{nome_modelo_fp32}.pth"
    
    if not os.path.exists(caminho_pesos):
        print(f"\n[AVISO] Pesos não encontrados em '{caminho_pesos}'. Pulando quantização deste modelo.")
        continue
        
    print(f" Processing: {nome_modelo_fp32} -> Quantização INT8")
    
    pasta_saida_quant = f"Resultados_quantizacao/{nome_modelo_fp32}_INT8"
    os.makedirs(pasta_saida_quant, exist_ok=True)
    
    modelo_fp32_calib = MODELO_CLASSE(in_channels=len(PRODUTOS_ENTRADA), out_channels=1).to(DEVICE_QUANT)
    modelo_fp32_calib.load_state_dict(torch.load(caminho_pesos, map_location=DEVICE_QUANT, weights_only=True))
    modelo_fp32_calib.eval()
    
    quantizer_calib = torch_quantizer(
        quant_mode='calib',
        module=modelo_fp32_calib,
        input_args=(dummy_input,),
        output_dir=pasta_saida_quant,
        device=DEVICE_QUANT
    )
    quant_model_calib = quantizer_calib.quant_model
    
    print("--> Rodando passagem de calibração PTQ (150 amostras)...")
    with torch.no_grad():
        for batch in dataloader_calib:
            inputs = normalizador_calib(batch["input"].to(DEVICE_QUANT))
            _ = quant_model_calib(inputs)
            
    quantizer_calib.export_quant_config()
    print(f"--> Tabela de quantização salva em: '{pasta_saida_quant}/quant_info.json'")
    
    modelo_fp32_test = MODELO_CLASSE(in_channels=len(PRODUTOS_ENTRADA), out_channels=1).to(DEVICE_QUANT)
    modelo_fp32_test.load_state_dict(torch.load(caminho_pesos, map_location=DEVICE_QUANT, weights_only=True))
    modelo_fp32_test.eval()
    
    quantizer_test = torch_quantizer(
        quant_mode='test',
        module=modelo_fp32_test,
        input_args=(dummy_input,),
        output_dir=pasta_saida_quant,
        device=DEVICE_QUANT
    )
    quant_model_int8 = quantizer_test.quant_model
    
    nome_modelo_salvar_int8 = f"{NOME_MODELO}_INT8_mag1c_rgb"
    
    print("\n--> [1/2] Avaliando modelo quantizado em GPU (CUDA)...")
    avaliar_e_visualizar(
        modelo_escolhido=lambda **kw: quant_model_int8,
        nome_modelo_salvo=nome_modelo_salvar_int8,
        produtos_entrada=PRODUTOS_ENTRADA,
        device='cuda'
    )
    
    print("\n--> [2/2] Avaliando modelo quantizado em CPU...")
    avaliar_e_visualizar(
        modelo_escolhido=lambda **kw: quant_model_int8,
        nome_modelo_salvo=nome_modelo_salvar_int8,
        produtos_entrada=PRODUTOS_ENTRADA,
        device='cpu'
    )